# SPANet direct `.pt2` vs baseline taggers

This notebook compares two jet-assignment strategies against the same truth definition from the SPANet training code:

- `baseline`: existing `HHBJet` and `VBFJet` collections from the tagger-based selection
- `direct_pt2_SPANet`: direct inference from the exported `spanet_5features_dynamic.pt2` on `SPANetJet`

The notebook is streamed file-by-file to avoid kernel crashes.

In [1]:
import gc
import sys
from glob import glob
from pathlib import Path

import awkward as ak
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

PARQUET_GLOB = "/local/norman/UHHAnalysis/test_tt/spanet_v7/events_*.parquet" if Path("/local/norman/UHHAnalysis/test_tt/spanet_v7").exists() else "/home/norman/UHHAnalysis/test_tt/spanet_v7/events_*.parquet"
SPANET_REPO = "/home/norman/SPANet" if Path("/home/norman/SPANet").exists() else "/home/norman/UHHAnalysis/SPANet"
MAX_JETS = 16
FEATURES = ("mass", "pt", "eta", "phi", "btagDeepFlavB")
BATCH_SIZE = 512
UINT64_MISSING = np.iinfo(np.uint64).max

REQUIRED_COLUMNS = [
    "SPANetJet.mass",
    "SPANetJet.pt",
    "SPANetJet.eta",
    "SPANetJet.phi",
    "SPANetJet.btagDeepFlavB",
    "SPANetJet.deterministic_seed",
    "HHBJet.deterministic_seed",
    "VBFJet.deterministic_seed",
    "Jet.deterministic_seed",
    "Jet.pt",
    "Jet.eta",
    "Jet.phi",
    "GenPart.pt",
    "GenPart.eta",
    "GenPart.phi",
    "GenPart.mass",
    "GenPart.pdgId",
    "GenPart.status",
    "GenPart.genPartIdxMother",
    "GenJet.pt",
    "GenJet.eta",
    "GenJet.phi",
    "GenJet.mass",
    "LHEPart.status",
    "LHEPart.pdgId",
    "LHEPart.pt",
    "LHEPart.eta",
    "LHEPart.phi",
    "LHEPart.mass",
]

MODEL_CANDIDATES = [
    "/local/norman/UHHAnalysis/SPANet/outputs/260706_SPANet_22v1_anygpu/spanet_5features_dynamic.pt2",
    "/home/norman/SPANet/outputs/260706_SPANet_22v1_anygpu/spanet_5features_dynamic.pt2",
    "/home/norman/UHHAnalysis/spanet_5features_dynamic.pt2",
    "/home/norman/UHHAnalysis/test_tt/spanet_5features_dynamic.pt2",
    "/local/norman/UHHAnalysis/test_tt/spanet_5features_dynamic.pt2",
]

MODEL_PATH = next((path for path in MODEL_CANDIDATES if Path(path).exists()), None)
if MODEL_PATH is None:
    raise FileNotFoundError(
        "Could not find spanet_5features_dynamic.pt2. Checked: " + str(MODEL_CANDIDATES)
    )

if SPANET_REPO not in sys.path:
    sys.path.append(SPANET_REPO)

files = sorted(glob(PARQUET_GLOB))
if not files:
    raise FileNotFoundError(f"No parquet files found for pattern: {PARQUET_GLOB}")

print("files:", len(files))
for file in files:
    print(" -", file)
print("MODEL_PATH:", MODEL_PATH)

files: 6
 - /local/norman/UHHAnalysis/test_tt/spanet_v7/events_0.parquet
 - /local/norman/UHHAnalysis/test_tt/spanet_v7/events_1.parquet
 - /local/norman/UHHAnalysis/test_tt/spanet_v7/events_2.parquet
 - /local/norman/UHHAnalysis/test_tt/spanet_v7/events_3.parquet
 - /local/norman/UHHAnalysis/test_tt/spanet_v7/events_4.parquet
 - /local/norman/UHHAnalysis/test_tt/spanet_v7/events_5.parquet
MODEL_PATH: /local/norman/UHHAnalysis/SPANet/outputs/260706_SPANet_22v1_anygpu/spanet_5features_dynamic.pt2


In [2]:
import torch

from spanet.network.prediction_selection import extract_predictions
from utils.convert_hh_bbtautau_parquet import higgs_bb_truth, matched_truth_indices, vbf_targets

print("torch version:", torch.__version__)
module = torch.export.load(MODEL_PATH).module()
print("loaded export")

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch version: 2.8.0+cu128


W0721 13:04:53.027000 2147670 site-packages/torch/_export/serde/serialize.py:2799] Symbol u0 did not appear in the graph that was deserialized


loaded export


In [3]:
def pad_feature(field, max_jets, fill_value=0.0):
    return ak.to_numpy(ak.fill_none(ak.pad_none(field, max_jets, axis=1, clip=True), fill_value))


def pad_bool_mask(field, max_jets):
    return ak.to_numpy(ak.fill_none(ak.pad_none(field, max_jets, axis=1, clip=True), False))


def pad_uint64_feature(field, max_jets, fill_value=UINT64_MISSING):
    arr = ak.pad_none(field, max_jets, axis=1, clip=True)
    arr = ak.fill_none(arr, np.uint64(fill_value))
    return ak.to_numpy(arr)


def pad2_uint64(field):
    return pad_uint64_feature(field, 2, fill_value=UINT64_MISSING)


def has_complete_pair(seed_pairs):
    return np.all(seed_pairs != UINT64_MISSING, axis=1)


def exact_pair_match(selected_pairs, truth_pairs):
    selected_sorted = np.sort(selected_pairs, axis=1)
    truth_sorted = np.sort(truth_pairs, axis=1)
    return has_complete_pair(selected_pairs) & has_complete_pair(truth_pairs) & np.all(selected_sorted == truth_sorted, axis=1)


def correct_jet_count(selected_pairs, truth_pairs):
    return (selected_pairs[:, :, None] == truth_pairs[:, None, :]).any(axis=2).sum(axis=1)


def take_seed_pair_from_indices(jets, indices):
    max_jets = int(ak.max(ak.num(jets.deterministic_seed, axis=1)))
    jet_seeds = pad_uint64_feature(jets.deterministic_seed, max_jets, fill_value=UINT64_MISSING)
    safe = indices.copy()
    safe[safe < 0] = 0
    gathered = jet_seeds[np.arange(len(jet_seeds))[:, None], safe]
    gathered[np.any(indices < 0, axis=1)] = UINT64_MISSING
    return gathered


def take_seed_pair_from_local_indices(spanetjet_seeds, indices):
    safe = indices.copy()
    safe[safe < 0] = 0
    gathered = spanetjet_seeds[np.arange(len(spanetjet_seeds))[:, None], safe]
    gathered[np.any(indices < 0, axis=1)] = UINT64_MISSING
    return gathered


def init_pair_accumulator(name, target):
    return {
        "algorithm": name,
        "target": target,
        "truth_ready_events": 0,
        "events_with_2_selected": 0,
        "selected_jets_total": 0,
        "selected_true_jets": 0,
        "exact_pair_match_events": 0,
        "correct_jets_0": 0,
        "correct_jets_1": 0,
        "correct_jets_2": 0,
    }


def update_pair_accumulator(acc, selected_pairs, truth_pairs, truth_valid_mask):
    selected_pairs = selected_pairs[truth_valid_mask]
    truth_pairs = truth_pairs[truth_valid_mask]
    acc["truth_ready_events"] += len(selected_pairs)

    has2 = has_complete_pair(selected_pairs)
    acc["events_with_2_selected"] += int(has2.sum())

    valid_selected = selected_pairs != UINT64_MISSING
    acc["selected_jets_total"] += int(valid_selected.sum())
    acc["selected_true_jets"] += int(((selected_pairs[:, :, None] == truth_pairs[:, None, :]).any(axis=2) & valid_selected).sum())

    exact = exact_pair_match(selected_pairs, truth_pairs)
    acc["exact_pair_match_events"] += int(exact.sum())

    counts = correct_jet_count(selected_pairs, truth_pairs)
    for n in [0, 1, 2]:
        acc[f"correct_jets_{n}"] += int((counts == n).sum())


def finalize_pair_accumulator(acc):
    truth_ready = acc["truth_ready_events"]
    selected_events = acc["events_with_2_selected"]
    selected_jets_total = acc["selected_jets_total"]
    exact = acc["exact_pair_match_events"]
    return {
        "algorithm": acc["algorithm"],
        "target": acc["target"],
        "truth_ready_events": truth_ready,
        "events_with_2_selected": selected_events,
        "selected_pair_fraction_pct": 100.0 * selected_events / truth_ready if truth_ready else np.nan,
        "selected_true_jet_fraction_pct": 100.0 * acc["selected_true_jets"] / selected_jets_total if selected_jets_total else np.nan,
        "exact_pair_match_events": exact,
        "exact_pair_match_pct": 100.0 * exact / truth_ready if truth_ready else np.nan,
        "exact_pair_match_pct_given_2_selected": 100.0 * exact / selected_events if selected_events else np.nan,
    }


def fraction_rows(acc):
    truth_ready = acc["truth_ready_events"]
    rows = []
    for n in [0, 1, 2]:
        rows.append({
            "algorithm": acc["algorithm"],
            "target": acc["target"],
            "correct_jets": n,
            "fraction_pct": 100.0 * acc[f"correct_jets_{n}"] / truth_ready if truth_ready else np.nan,
        })
    return rows


def init_all4_accumulator(name):
    return {
        "algorithm": name,
        "truth_ready_events": 0,
        "events_with_4_selected": 0,
        "all4_exact_events": 0,
    }


def update_all4_accumulator(acc, hhb_pairs, vbf_pairs, truth_hhb_pairs, truth_vbf_pairs, truth_all4_mask):
    hhb_pairs = hhb_pairs[truth_all4_mask]
    vbf_pairs = vbf_pairs[truth_all4_mask]
    truth_hhb_pairs = truth_hhb_pairs[truth_all4_mask]
    truth_vbf_pairs = truth_vbf_pairs[truth_all4_mask]

    acc["truth_ready_events"] += len(hhb_pairs)
    full_selected = has_complete_pair(hhb_pairs) & has_complete_pair(vbf_pairs)
    acc["events_with_4_selected"] += int(full_selected.sum())

    all4_exact = exact_pair_match(hhb_pairs, truth_hhb_pairs) & exact_pair_match(vbf_pairs, truth_vbf_pairs)
    acc["all4_exact_events"] += int(all4_exact.sum())


def finalize_all4_accumulator(acc):
    truth_ready = acc["truth_ready_events"]
    selected = acc["events_with_4_selected"]
    exact = acc["all4_exact_events"]
    return {
        "algorithm": acc["algorithm"],
        "truth_ready_events": truth_ready,
        "events_with_4_selected": selected,
        "full_selection_fraction_pct": 100.0 * selected / truth_ready if truth_ready else np.nan,
        "all4_exact_events": exact,
        "all4_exact_pct": 100.0 * exact / truth_ready if truth_ready else np.nan,
        "all4_exact_pct_given_4_selected": 100.0 * exact / selected if selected else np.nan,
    }

In [4]:
HHB_DR = 0.4
VBF_TRUTH = "genjet"
LHE_TO_GEN_DR = 0.4
GEN_TO_RECO_DR = 0.3
USE_VBF_OVERLAP_CLEANING = True

pair_accumulators = {
    ("baseline", "HHB"): init_pair_accumulator("baseline", "HHB"),
    ("baseline", "VBF"): init_pair_accumulator("baseline", "VBF"),
    ("direct_pt2_SPANet", "HHB"): init_pair_accumulator("direct_pt2_SPANet", "HHB"),
    ("direct_pt2_SPANet", "VBF"): init_pair_accumulator("direct_pt2_SPANet", "VBF"),
}

all4_accumulators = {
    "baseline": init_all4_accumulator("baseline"),
    "direct_pt2_SPANet": init_all4_accumulator("direct_pt2_SPANet"),
}

truth_event_count = 0
truth_hhb_valid_count = 0
truth_vbf_valid_count = 0
truth_all4_valid_count = 0

for file_index, file in enumerate(files, start=1):
    print(f"[{file_index}/{len(files)}] {file}")
    print("   events:", ak.metadata_from_parquet(file)["num_rows"])
    events = ak.from_parquet(file, columns=REQUIRED_COLUMNS)

    spanetjet_feature_arrays = [pad_feature(events.SPANetJet[feature], MAX_JETS)[..., None] for feature in FEATURES]
    features = np.concatenate(spanetjet_feature_arrays, axis=2).astype(np.float32)
    mask = pad_bool_mask(events.SPANetJet.pt > -1.0e8, MAX_JETS)
    spanetjet_seeds = pad_uint64_feature(events.SPANetJet.deterministic_seed, MAX_JETS, fill_value=UINT64_MISSING)

    direct_hhb_batches = []
    direct_vbf_batches = []
    for start in range(0, len(features), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(features))
        batch_features = torch.from_numpy(features[start:stop])
        batch_mask = torch.from_numpy(mask[start:stop])
        with torch.no_grad():
            assignment_logits, detection_probs = module(batch_features, batch_mask)

        assignment_logits_np = [np.nan_to_num(t.detach().cpu().numpy(), nan=-np.inf) for t in assignment_logits]
        predictions = extract_predictions(assignment_logits_np)
        direct_hhb_batches.append(predictions[0])
        direct_vbf_batches.append(predictions[1])

    direct_hhb_indices = np.concatenate(direct_hhb_batches, axis=0)
    direct_vbf_indices = np.concatenate(direct_vbf_batches, axis=0)
    direct_hhb_seeds = take_seed_pair_from_local_indices(spanetjet_seeds, direct_hhb_indices)
    direct_vbf_seeds = take_seed_pair_from_local_indices(spanetjet_seeds, direct_vbf_indices)

    baseline_hhb_seeds = pad2_uint64(events.HHBJet.deterministic_seed)
    baseline_vbf_seeds = pad2_uint64(events.VBFJet.deterministic_seed)

    truth_b1, truth_b2 = matched_truth_indices(higgs_bb_truth(events), events.Jet, HHB_DR)
    truth_q1, truth_q2 = vbf_targets(
        events,
        "Jet",
        VBF_TRUTH,
        (truth_b1, truth_b2),
        LHE_TO_GEN_DR,
        GEN_TO_RECO_DR,
        USE_VBF_OVERLAP_CLEANING,
    )

    truth_hhb_jet_indices = np.column_stack([truth_b1, truth_b2])
    truth_vbf_jet_indices = np.column_stack([truth_q1, truth_q2])
    truth_hhb_valid = np.all(truth_hhb_jet_indices >= 0, axis=1)
    truth_vbf_valid = np.all(truth_vbf_jet_indices >= 0, axis=1)
    truth_all4_valid = truth_hhb_valid & truth_vbf_valid

    truth_hhb_seeds = take_seed_pair_from_indices(events.Jet, truth_hhb_jet_indices)
    truth_vbf_seeds = take_seed_pair_from_indices(events.Jet, truth_vbf_jet_indices)

    truth_event_count += len(events)
    truth_hhb_valid_count += int(truth_hhb_valid.sum())
    truth_vbf_valid_count += int(truth_vbf_valid.sum())
    truth_all4_valid_count += int(truth_all4_valid.sum())

    update_pair_accumulator(pair_accumulators[("baseline", "HHB")], baseline_hhb_seeds, truth_hhb_seeds, truth_hhb_valid)
    update_pair_accumulator(pair_accumulators[("baseline", "VBF")], baseline_vbf_seeds, truth_vbf_seeds, truth_vbf_valid)
    update_pair_accumulator(pair_accumulators[("direct_pt2_SPANet", "HHB")], direct_hhb_seeds, truth_hhb_seeds, truth_hhb_valid)
    update_pair_accumulator(pair_accumulators[("direct_pt2_SPANet", "VBF")], direct_vbf_seeds, truth_vbf_seeds, truth_vbf_valid)

    update_all4_accumulator(all4_accumulators["baseline"], baseline_hhb_seeds, baseline_vbf_seeds, truth_hhb_seeds, truth_vbf_seeds, truth_all4_valid)
    update_all4_accumulator(all4_accumulators["direct_pt2_SPANet"], direct_hhb_seeds, direct_vbf_seeds, truth_hhb_seeds, truth_vbf_seeds, truth_all4_valid)

    del events
    del spanetjet_feature_arrays, features, mask, spanetjet_seeds
    del direct_hhb_batches, direct_vbf_batches, direct_hhb_indices, direct_vbf_indices
    del direct_hhb_seeds, direct_vbf_seeds, baseline_hhb_seeds, baseline_vbf_seeds
    del truth_b1, truth_b2, truth_q1, truth_q2, truth_hhb_jet_indices, truth_vbf_jet_indices
    del truth_hhb_valid, truth_vbf_valid, truth_all4_valid, truth_hhb_seeds, truth_vbf_seeds
    gc.collect()

truth_summary = pd.DataFrame({
    "events": [truth_event_count],
    "truth_hhb_valid_pct": [100.0 * truth_hhb_valid_count / truth_event_count],
    "truth_vbf_valid_pct": [100.0 * truth_vbf_valid_count / truth_event_count],
    "truth_all4_valid_pct": [100.0 * truth_all4_valid_count / truth_event_count],
})
truth_summary

[1/6] /local/norman/UHHAnalysis/test_tt/spanet_v7/events_0.parquet
   events: 30244
[2/6] /local/norman/UHHAnalysis/test_tt/spanet_v7/events_1.parquet
   events: 33367
[3/6] /local/norman/UHHAnalysis/test_tt/spanet_v7/events_2.parquet
   events: 35463
[4/6] /local/norman/UHHAnalysis/test_tt/spanet_v7/events_3.parquet
   events: 21037
[5/6] /local/norman/UHHAnalysis/test_tt/spanet_v7/events_4.parquet
   events: 31999
[6/6] /local/norman/UHHAnalysis/test_tt/spanet_v7/events_5.parquet
   events: 29159


,events,truth_hhb_valid_pct,truth_vbf_valid_pct,truth_all4_valid_pct
0,181269,54.196801,9.014779,4.032681


In [5]:
target_summary = pd.DataFrame([
    finalize_pair_accumulator(pair_accumulators[("baseline", "HHB")]),
    finalize_pair_accumulator(pair_accumulators[("baseline", "VBF")]),
    finalize_pair_accumulator(pair_accumulators[("direct_pt2_SPANet", "HHB")]),
    finalize_pair_accumulator(pair_accumulators[("direct_pt2_SPANet", "VBF")]),
])

target_summary

,algorithm,target,truth_ready_events,events_with_2_selected,selected_pair_fraction_pct,selected_true_jet_fraction_pct,exact_pair_match_events,exact_pair_match_pct,exact_pair_match_pct_given_2_selected
0,baseline,HHB,98242,98242,100.000000,98.255329,94891,96.589035,96.589035
1,baseline,VBF,16341,10961,67.076678,83.308321,7231,44.250658,65.970258
2,direct_pt2_SPANet,HHB,98242,98242,100.000000,98.198836,92047,93.694143,93.694143
3,direct_pt2_SPANet,VBF,16341,15822,96.823940,93.202503,12404,75.907227,78.397168


In [6]:
all4_summary = pd.DataFrame([
    finalize_all4_accumulator(all4_accumulators["baseline"]),
    finalize_all4_accumulator(all4_accumulators["direct_pt2_SPANet"]),
])

all4_summary

,algorithm,truth_ready_events,events_with_4_selected,full_selection_fraction_pct,all4_exact_events,all4_exact_pct,all4_exact_pct_given_4_selected
0,baseline,7310,7007,95.854993,5848,80.000000,83.459398
1,direct_pt2_SPANet,7310,7310,100.000000,6627,90.656635,90.656635


In [7]:
fraction_table = pd.DataFrame(
    fraction_rows(pair_accumulators[("baseline", "HHB")]) +
    fraction_rows(pair_accumulators[("baseline", "VBF")]) +
    fraction_rows(pair_accumulators[("direct_pt2_SPANet", "HHB")]) +
    fraction_rows(pair_accumulators[("direct_pt2_SPANet", "VBF")])
)

fraction_table

,algorithm,target,correct_jets,fraction_pct
0,baseline,HHB,0,0.078378
1,baseline,HHB,1,3.332587
2,baseline,HHB,2,96.589035
3,baseline,VBF,0,8.365461
4,baseline,VBF,1,47.383881
5,baseline,VBF,2,44.250658
6,direct_pt2_SPANet,HHB,0,0.156756
7,direct_pt2_SPANet,HHB,1,3.288817
8,direct_pt2_SPANet,HHB,2,96.554427
9,direct_pt2_SPANet,VBF,0,3.677865
